# M11.5 · Modelden bağımsız kontrol — SARSA, Q-öğrenme, keşif

Değerlendirmeden kontrole: V yerine Q(s,a) öğren, çünkü açgözlü politikayı Q'dan model olmadan çıkarabiliriz. Monte Carlo kontrol (keşifçi başlangıçlar), politika üstü SARSA, politika dışı Q-öğrenme, beklenen SARSA, maksimizasyon yanlılığı ve çift Q-öğrenme; keşif ve sömürü ikilemi (ε-açgözlü, Boltzmann, iyimser başlangıç). Saf Python'da uçurum yürüyüşü, NumPy'da 10 000 deneyde maksimizasyon yanlılığı, PyTorch'ta eşzamanlı Q-öğrenmenin Q*'a yakınsaması ve 1000 paralel ortamda kontrol.

Etkileşimli sürüm: [https://m-gonul.github.io/pytorch-lab/m11-5-sarsa-q.html](https://m-gonul.github.io/pytorch-lab/m11-5-sarsa-q.html)

**Nasıl çalışılır:** her aşamada önce `%%writefile` hücresindeki kodu düzenleyip çalıştır (dosyaya yazar), sonra altındaki **Kontrol** hücresini çalıştır. Kontroller sitedekiyle birebir aynı. GPU gerekmez; Faz 7 modülleri hariç.

In [ ]:
#@title Kurulum: kontrol kütüphanesi (çalıştır, düzenleme)
%%writefile labcheck.py
"""labcheck — pytorch-lab'in ortak kontrol çekirdeği.

Aynı dosya üç yerde birebir çalışır:
  * tarayıcıda (Pyodide, Web Worker içinde),
  * yerel runner'da (uv run runner/torchlab.py),
  * CI'da ve Colab notebook'larında.

Protokol: öğrenci kodu `exec` ile ayrı bir namespace'te çalıştırılır, ardından
aşamanın kontrol dosyası (`check(ns, R)` fonksiyonu tanımlar) aynı namespace
üzerinde çalışır. Sonuç JSON'a çevrilebilir bir dict olarak döner.
"""
from __future__ import annotations

import ast
import contextlib
import io
import json
import math
import time
import traceback

__all__ = ["Report", "run_stage", "close", "to_list", "no_loops", "no_imports",
           "uses_any", "need", "ensure", "bench", "fmt_time", "loop_depth", "max_loops"]


# --------------------------------------------------------------------------- yardımcılar
def to_list(x):
    """Tensor / ndarray / liste / skaleri iç içe Python listesine (float) çevirir."""
    if hasattr(x, "detach"):  # torch.Tensor
        x = x.detach().cpu()
        x = x.tolist()
    elif hasattr(x, "tolist"):  # numpy
        x = x.tolist()
    if isinstance(x, (list, tuple)):
        return [to_list(v) for v in x]
    if isinstance(x, bool):
        return float(x)
    if isinstance(x, (int, float)):
        return float(x)
    return x


def _shape(x):
    if isinstance(x, list):
        return (len(x),) + (_shape(x[0]) if x else ())
    return ()


def _flat(x):
    if isinstance(x, list):
        for v in x:
            yield from _flat(v)
    else:
        yield x


def close(got, want, rtol=1e-5, atol=1e-6, what="sonuç"):
    """`got` ile `want` aynı shape'te ve sayısal olarak yakın mı? Değilse AssertionError."""
    g, w = to_list(got), to_list(want)
    if _shape(g) != _shape(w):
        raise AssertionError(f"{what}: shape {_shape(g)} bekleniyordu {_shape(w)}")
    worst = 0.0
    for a, b in zip(_flat(g), _flat(w)):
        if not isinstance(a, float):
            raise AssertionError(f"{what}: sayı olmayan değer {a!r}")
        if math.isnan(a) and math.isnan(b):
            continue
        err = abs(a - b)
        if err > atol + rtol * abs(b):
            raise AssertionError(f"{what}: {a:.6g} bulundu, {b:.6g} bekleniyordu")
        worst = max(worst, err)
    return worst


def _tree(src):
    try:
        return ast.parse(src)
    except SyntaxError:
        return None


def no_loops(src, allow_comprehension=False):
    """Kaynakta `for`/`while` döngüsü (ve istenirse comprehension) var mı? Varsa AssertionError."""
    t = _tree(src)
    if t is None:
        return
    banned = (ast.For, ast.While, ast.AsyncFor)
    if not allow_comprehension:
        banned += (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)
    for node in ast.walk(t):
        if isinstance(node, banned):
            kind = "döngü" if isinstance(node, (ast.For, ast.While, ast.AsyncFor)) else "comprehension"
            raise AssertionError(f"satır {node.lineno}: bu aşamada {kind} yasak — işlemi vektörel yaz")


def _func(tree, name):
    for node in ast.walk(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) and node.name == name:
            return node
    return None


def loop_depth(src, func=None):
    """Kaynakta (ya da yalnız `func` fonksiyonunda) iç içe for/while derinliği. Comprehension'lar da sayılır."""
    t = _tree(src)
    if t is None:
        return 0
    if func is not None:
        t = _func(t, func)
        if t is None:
            raise AssertionError(f"`{func}` fonksiyonu bulunamadı")
    loops = (ast.For, ast.While, ast.AsyncFor, ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)

    def depth(node):
        own = 1 if isinstance(node, loops) else 0
        if isinstance(node, (ast.ListComp, ast.SetComp, ast.DictComp, ast.GeneratorExp)):
            own = len(node.generators)
        return own + max((depth(c) for c in ast.iter_child_nodes(node)), default=0)
    return depth(t)


def max_loops(src, n, func=None, why=""):
    """Döngü derinliği en fazla n mi? (ör. eğitim adımları için tek döngü serbest, örnekler üzerinde döngü yasak)"""
    d = loop_depth(src, func)
    if d > n:
        where = f"`{func}` içinde " if func else ""
        raise AssertionError(f"{where}{d} katmanlı döngü var; en fazla {n} olmalı{(' — ' + why) if why else ''}")


def no_imports(src, names):
    """Belirtilen modüllerin import edilmediğini doğrular (ör. b aşamasında numpy yok)."""
    t = _tree(src)
    if t is None:
        return
    for node in ast.walk(t):
        mods = []
        if isinstance(node, ast.Import):
            mods = [a.name.split(".")[0] for a in node.names]
        elif isinstance(node, ast.ImportFrom) and node.module:
            mods = [node.module.split(".")[0]]
        for m in mods:
            if m in names:
                raise AssertionError(f"satır {node.lineno}: bu aşamada `{m}` kullanılmamalı")


def uses_any(src, words):
    """Kaynak, verilen isimlerden en az birini (attribute ya da isim olarak) kullanıyor mu?"""
    t = _tree(src)
    if t is None:
        return False
    found = set()
    for node in ast.walk(t):
        if isinstance(node, ast.Attribute):
            found.add(node.attr)
        elif isinstance(node, ast.Name):
            found.add(node.id)
    return any(w in found for w in words)


def ensure(cond, msg):
    """`cond` yanlışsa `msg` ile AssertionError."""
    if not cond:
        raise AssertionError(msg)


def need(ns, name, kind="fonksiyon"):
    if name not in ns:
        raise AssertionError(f"`{name}` adlı {kind} tanımlanmamış")
    return ns[name]


def fmt_time(s):
    if s < 1e-6:
        return f"{s * 1e9:.0f} ns"
    if s < 1e-3:
        return f"{s * 1e6:.1f} µs"
    if s < 1:
        return f"{s * 1e3:.2f} ms"
    return f"{s:.2f} s"


def bench(fn, repeat=5, number=None, budget=0.25):
    """`fn()`'in medyan süresi (saniye). number verilmezse ~budget saniyeye göre ayarlanır."""
    fn()  # ısınma (lazy init, torch.compile vb.)
    if number is None:
        t0 = time.perf_counter()
        fn()
        one = max(time.perf_counter() - t0, 1e-7)
        number = max(1, min(10_000, int(budget / one / repeat)))
    times = []
    for _ in range(repeat):
        t0 = time.perf_counter()
        for _ in range(number):
            fn()
        times.append((time.perf_counter() - t0) / number)
    times.sort()
    return times[len(times) // 2]


# --------------------------------------------------------------------------- rapor
class Report:
    def __init__(self):
        self.items = []    # {"name", "ok", "msg"}
        self.tables = []   # {"title", "cols", "rows", "note"}
        self.notes = []

    def test(self, name, fn=None):
        """`R.test("ad", lambda: ...)` ya da dekoratör olarak `@R.test("ad")`."""
        if fn is None:
            return lambda f: (self.test(name, f), f)[1]
        try:
            res = fn()
            msg = res if isinstance(res, str) else ""
            self.items.append({"name": name, "ok": True, "msg": msg})
        except AssertionError as e:
            self.items.append({"name": name, "ok": False, "msg": str(e) or "koşul sağlanmadı"})
        except Exception as e:  # öğrenci kodu patladı
            tb = traceback.extract_tb(e.__traceback__)
            where = next((f"satır {f.lineno}: " for f in reversed(tb) if f.filename == "<öğrenci>"), "")
            self.items.append({"name": name, "ok": False, "msg": f"{where}{type(e).__name__}: {e}"})
        return fn

    def table(self, title, cols, rows, note=""):
        self.tables.append({"title": title, "cols": list(cols), "rows": [list(map(str, r)) for r in rows], "note": note})

    def note(self, text):
        self.notes.append(str(text))

    @property
    def ok(self):
        return bool(self.items) and all(i["ok"] for i in self.items)


def _short_tb(e):
    tb = traceback.extract_tb(e.__traceback__)
    frames = [f for f in tb if f.filename == "<öğrenci>"]
    loc = f"satır {frames[-1].lineno}: " if frames else ""
    return f"{loc}{type(e).__name__}: {e}"


def run_stage(code, check_code=None, max_out=20_000):
    """Öğrenci kodunu çalıştırır, istenirse kontrolleri uygular. JSON'a çevrilebilir dict döner."""
    out = io.StringIO()
    ns = {"__name__": "__main__", "__source__": code}
    t0 = time.perf_counter()
    error = None
    with contextlib.redirect_stdout(out), contextlib.redirect_stderr(out):
        try:
            exec(compile(code, "<öğrenci>", "exec"), ns)
        except SyntaxError as e:
            error = f"satır {e.lineno}: SyntaxError: {e.msg}"
        except Exception as e:
            error = _short_tb(e)
    elapsed = time.perf_counter() - t0
    R = Report()
    if check_code is not None and error is None:
        cns = {"__name__": "labcheck_stage"}
        with contextlib.redirect_stdout(out):
            try:
                exec(compile(check_code, "<kontrol>", "exec"), cns)
                cns["check"](ns, R)
            except Exception as e:  # kontrol dosyası hatası: kullanıcıya göster
                R.items.append({"name": "kontrol çalıştırılamadı", "ok": False,
                                "msg": f"{type(e).__name__}: {e}"})
    text = out.getvalue()
    if len(text) > max_out:
        text = text[:max_out] + f"\n… (çıktı {len(text) - max_out} karakter kısaltıldı)"
    return {
        "stdout": text,
        "error": error,
        "elapsed": elapsed,
        "checked": check_code is not None,
        "ok": error is None and (check_code is None or R.ok),
        "items": R.items,
        "tables": R.tables,
        "notes": R.notes,
    }


def run_stage_json(code, check_code=None):
    return json.dumps(run_stage(code, check_code), ensure_ascii=False)


def run_file(path, check_code):
    """Colab/Jupyter yardımcısı: `%%writefile` ile yazılmış aşama dosyasını kontrol eder ve özetler."""
    with open(path, encoding="utf-8") as f:
        res = run_stage(f.read(), check_code)
    if res["stdout"].strip():
        print(res["stdout"].rstrip())
    if res["error"]:
        print("✗ hata:", res["error"])
    for it in res["items"]:
        print(("✓ " if it["ok"] else "✗ ") + it["name"] + (f" — {it['msg']}" if it["msg"] else ""))
    for t in res["tables"]:
        print(f"\n{t['title']}")
        w = [max(len(str(c)), *(len(r[i]) for r in t["rows"])) for i, c in enumerate(t["cols"])]
        print("  ".join(str(c).ljust(w[i]) for i, c in enumerate(t["cols"])))
        for r in t["rows"]:
            print("  ".join(v.ljust(w[i]) for i, v in enumerate(r)))
        if t["note"]:
            print(t["note"])
    print("\nSONUÇ:", "geçti ✓" if res["ok"] else "henüz değil ✗")
    return res["ok"]

In [ ]:
#@title Kurulum: bu modülün kontrolleri (çalıştır, düzenleme)
%%writefile checks_m11_5_sarsa_q.py
CHECKS = {
 "b": "from labcheck import close, no_imports, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"numpy kullanılmıyor\", lambda: no_imports(src, {\"numpy\", \"torch\"}))\n\n    def env():\n        f = need(ns, \"cliff_step\")\n        ensure(f(36, 1) == (36, -100.0, False), \"başlangıçtan sağa: uçurum\")\n        ensure(f(36, 0) == (24, -1.0, False) and f(36, 2) == (36, -1.0, False) and f(36, 3) == (36, -1.0, False), \"yukarı / sınırlar\")\n        ensure(f(35, 2) == (47, -1.0, True), \"hedef\")\n        ensure(f(0, 0) == (0, -1.0, False), \"köşe\")\n    R_.test(\"cliff_step\", env)\n\n    def online():\n        S = Qm = 0\n        for seed in range(10):\n            S += sum(need(ns, \"sarsa\")(500, 0.5, 0.1, seed)[1][-100:]) / 100\n            Qm += sum(need(ns, \"qlearning\")(500, 0.5, 0.1, seed)[1][-100:]) / 100\n        S /= 10; Qm /= 10\n        ensure(S > Qm + 5, f\"eğitim sırasında SARSA daha çok ödül toplamalı ({S:.1f} vs {Qm:.1f})\")\n        gq = [need(ns, \"greedy_return\")(ns[\"qlearning\"](500, 0.5, 0.1, s)[0]) for s in range(10)]\n        gs = [ns[\"greedy_return\"](ns[\"sarsa\"](1000, 0.1, 0.1, s)[0]) for s in range(10)]\n        ensure(sum(g == -13 for g in gq) >= 9, \"Q-öğrenme açgözlü yolu: kenardan, −13\")\n        ensure(sum(-21 <= g <= -15 for g in gs) >= 8, \"SARSA açgözlü yolu: güvenli, −15…−21\")\n        R_.table(\"Uçurum yürüyüşü, 10 deney\", [\"yöntem\", \"son 100 bölümde ort. getiri (α = 0.5)\", \"açgözlü yolun getirisi\"],\n                 [[\"SARSA\", f\"{S:.1f}\", \", \".join(str(int(g)) for g in gs)], [\"Q-öğrenme\", f\"{Qm:.1f}\", \", \".join(str(int(g)) for g in gq)]],\n                 note=\"Q-öğrenme optimal (kenar) yolu öğrenir ama ε-açgözlü davranırken uçuruma düşer; SARSA keşfi hesaba katarak uzak yolu seçer (Sutton Şekil 6.4). SARSA'nın açgözlü yolu α = 0.1, 1000 bölümle.\")\n    R_.test(\"SARSA ve Q-öğrenme\", online)\n",
 "c": "import numpy as np\nfrom labcheck import close, max_loops, need, ensure\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"yalnız bölüm döngüsü\", lambda: max_loops(src, 1, func=\"max_bias\"))\n\n    def p():\n        Q = np.array([[1.0, 3.0, 2.0], [0.0, -1.0, 5.0]])\n        close(need(ns, \"eps_greedy_probs\")(Q, 0.3), np.array([[0.1, 0.8, 0.1], [0.1, 0.1, 0.8]]), what=\"ε-açgözlü olasılıklar\")\n    R_.test(\"eps_greedy_probs\", p)\n\n    def mb():\n        q = need(ns, \"max_bias\")(10000, 300, seed=1)\n        d = ns[\"max_bias\"](10000, 300, seed=1, double=True)\n        ensure(q.shape == (300,), \"şekil\")\n        ensure(q[5:30].max() > 0.75, \"Q-öğrenme başta solu fazlasıyla seçmeli\")\n        ensure(d.max() < 0.55 and d[30] < d[5], \"çift Q-öğrenme solu tercih etmemeli\")\n        ensure(d[-50:].mean() < 0.09 and q[-50:].mean() > d[-50:].mean(), \"çift Q optimal ε-açgözlü orana (%5) yaklaşmalı\")\n        R_.table(\"Sol seçme oranı, 10 000 deney (optimal: %5)\", [\"bölüm\", \"Q-öğrenme\", \"çift Q-öğrenme\"],\n                 [[e + 1, f\"{100 * q[e]:.1f}%\", f\"{100 * d[e]:.1f}%\"] for e in (0, 10, 20, 50, 100, 200, 299)],\n                 note=\"B'deki 10 eylemin gürültülü kestirimlerinden en büyüğü pozitif çıkar; Q-öğrenme bu yanlılığı A'ya taşır (Sutton Şekil 6.5).\")\n    R_.test(\"maksimizasyon yanlılığı\", mb)\n",
 "d": "import numpy as np\nimport torch\nfrom labcheck import close, need, ensure\n\n\ndef check(ns, R_):\n    g = torch.Generator().manual_seed(0)\n    S, A = 20, 4\n    P = torch.rand(A, S, S, generator=g, dtype=torch.float64) ** 4\n    P /= P.sum(-1, keepdim=True)\n    R = torch.rand(S, A, generator=g, dtype=torch.float64)\n\n    def qvi():\n        Q = need(ns, \"q_value_iteration\")(P, R, 0.9)\n        V = Q.max(1).values\n        ref = R + 0.9 * torch.stack([P[a] @ V for a in range(A)], 1)\n        close(Q.numpy(), ref.numpy(), atol=1e-8, what=\"Bellman optimallik denklemi\")\n        # aynı sonuç: politika değerlendirmesi ile\n        pi = Q.argmax(1)\n        Ppi = P[pi, torch.arange(S)]\n        Vpi = torch.linalg.solve(torch.eye(S, dtype=torch.float64) - 0.9 * Ppi, R[torch.arange(S), pi])\n        close(Vpi.numpy(), V.numpy(), atol=1e-8, what=\"V^{π*} = max Q*\")\n    R_.test(\"q_value_iteration\", qvi)\n\n    def sq():\n        Qs = ns[\"q_value_iteration\"](P, R, 0.9)\n        rows = []\n        errs = {}\n        for rate in (\"rescaled\", \"1/n\"):\n            for n in (100, 1000, 5000):\n                Q = need(ns, \"sync_q_learning\")(P, R, 0.9, n, rate, 1)\n                e = float((Q - Qs).abs().max())\n                errs[(rate, n)] = e\n                rows.append([rate, n, f\"{e:.3f}\", f\"{float((need(ns, 'greedy')(Q) == Qs.argmax(1)).double().mean()):.2f}\"])\n        ensure(errs[(\"rescaled\", 5000)] < 0.06, \"1/(1+(1−γ)n) ile Q* yakınına gelmeli\")\n        ensure(errs[(\"1/n\", 5000)] > 1.0, \"1/n ile hata büyük kalmalı\")\n        R_.table(\"Eşzamanlı Q-öğrenme, 20 durum × 4 eylem, γ = 0.9 (‖Q* ‖∞ ≈ %.1f)\" % float(Qs.abs().max()),\n                 [\"α_n\", \"yineleme\", \"‖Q − Q*‖∞\", \"açgözlü politika uyumu\"], rows,\n                 note=\"İkisi de Robbins–Monro koşullarını sağlar; ama 1/n ile önyükleme yanlılığı n^{−(1−γ)} hızında söner. Politika genelde değerlerden çok önce doğrulanır.\")\n    R_.test(\"sync_q_learning ve öğrenme oranları\", sq)\n",
 "e": "import random\nimport torch\nfrom labcheck import close, max_loops, need, ensure, bench, fmt_time\n\n\ndef loop_q(n_envs, n_steps, alpha=0.5, eps=0.1):\n    moves = [(-1, 0), (0, 1), (1, 0), (0, -1)]\n    for k in range(n_envs):\n        rng = random.Random(k)\n        Q = [[0.0] * 4 for _ in range(48)]\n        s = 36\n        for _ in range(n_steps):\n            q = Q[s]\n            a = rng.randrange(4) if rng.random() < eps else q.index(max(q))\n            r_, c_ = divmod(s, 12)\n            r_ = min(max(r_ + moves[a][0], 0), 3); c_ = min(max(c_ + moves[a][1], 0), 11)\n            if r_ == 3 and 1 <= c_ <= 10:\n                s2, r = 36, -100.0\n            else:\n                s2, r = r_ * 12 + c_, -1.0\n            Q[s][a] += alpha * (r + (0.0 if s2 == 47 else max(Q[s2])) - Q[s][a])\n            s = 36 if s2 == 47 else s2\n\n\ndef check(ns, R_):\n    src = ns[\"__source__\"]\n    R_.test(\"yalnız adım döngüsü\", lambda: (max_loops(src, 1, func=\"batched_control\"), max_loops(src, 1, func=\"greedy_returns\")))\n\n    def st():\n        s2, r, d = need(ns, \"step\")(torch.tensor([36, 36, 35, 0]), torch.tensor([1, 0, 2, 0]))\n        ensure(s2.tolist() == [36, 24, 47, 0] and r.tolist() == [-100, -1, -1, -1] and d.tolist() == [False, False, True, False], \"vektörel ortam\")\n    R_.test(\"step\", st)\n\n    def ctl():\n        rows = []\n        res = {}\n        for m in (\"sarsa\", \"expected\", \"q\"):\n            Q, rew = need(ns, \"batched_control\")(1000, 8000, 0.5, 0.1, m, 2)\n            G = need(ns, \"greedy_returns\")(Q)\n            res[m] = (float(rew[-3000:].mean()), G)\n            vals, cnt = torch.unique(G, return_counts=True)\n            rows.append([m, f\"{res[m][0]:.3f}\", \", \".join(f\"{int(v)}: {int(c)}\" for v, c in zip(vals, cnt))])\n        ensure(int((res[\"q\"][1] == -13).sum()) > 950, \"Q-öğrenme: kenar yolu\")\n        ensure(int(((res[\"expected\"][1] <= -15) & (res[\"expected\"][1] >= -19)).sum()) > 900, \"beklenen SARSA: güvenli yol\")\n        ensure(res[\"sarsa\"][0] > res[\"q\"][0] and res[\"expected\"][0] > res[\"q\"][0], \"politika üstü yöntemler eğitimde daha çok ödül toplar\")\n        t1 = bench(lambda: loop_q(20, 8000), repeat=1, number=1) * 50\n        t2 = bench(lambda: ns[\"batched_control\"](1000, 8000, 0.5, 0.1, \"q\"), repeat=1, number=1)\n        rows.append([\"süre: ortam döngüsü (saf Python, tahmini)\", fmt_time(t1), \"\"])\n        rows.append([\"süre: 1000 paralel ortam\", fmt_time(t2), \"\"])\n        R_.table(\"1000 ortam × 8000 adım, α = 0.5, ε = 0.1\", [\"yöntem\", \"son 3000 adımda ort. ödül/adım\", \"açgözlü getiri: ortam sayısı\"], rows,\n                 note=\"Beklenen SARSA, SARSA'nın a' örneklemesinden gelen varyansı yok eder: büyük α'da bile tüm ortamlarda aynı güvenli yolu bulur.\")\n    R_.test(\"SARSA, beklenen SARSA, Q-öğrenme\", ctl)\n",
 "x1": "from labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        f = need(ns, \"windy_step\")\n        ensure(f(3, 0, 1) == (3, 1) and f(3, 3, 1) == (2, 4) and f(3, 6, 1) == (1, 7) and f(0, 6, 0) == (0, 6), \"rüzgâr\")\n        res = [need(ns, \"sarsa_windy\")(8000, s) for s in range(8)]\n        lens = [need(ns, \"greedy_len\")(Q) for Q, _ in res]\n        eps = [h[-1] for _, h in res]\n        ensure(all(140 <= e <= 200 for e in eps), f\"8000 adımda ≈ 170 bölüm, bulunan {eps}\")\n        ensure(sum(l <= 18 for l in lens) >= 4 and min(lens) == 15, f\"açgözlü yol ≈ 15 adım, bulunan {lens}\")\n        h = res[0][1]\n        R_.table(\"Rüzgârlı ızgara, SARSA (Sutton Şekil 6.3)\", [\"zaman adımı\", \"biten bölüm\"], [[t, h[t - 1]] for t in (1000, 2000, 4000, 6000, 8000)],\n                 note=f\"Açgözlü yol uzunlukları (8 tohum): {lens}; en kısa yol 15. 100 = açgözlü politika döngüye girdi: hiç ziyaret edilmemiş durumların Q'su 0 kaldığı için (−1 ödüllere göre iyimser) saf açgözlü yürüyüş oraya sapıp duvara takılabilir; ε-açgözlü davranış politikası bu sorunu yaşamaz. Eğri başta yatay (ilk bölüm yüzlerce adım), sonra eğimi ≈ 1/17'ye oturur.\")\n    R_.test(\"rüzgârlı ızgara\", t)\n",
 "x2": "from labcheck import need, ensure\n\n\ndef check(ns, R_):\n    def t():\n        cfg = [(\"eps\", 0.0), (\"boltzmann\", 0.0), (\"greedy\", 0.0), (\"greedy\", 1.0), (\"greedy\", 2.0), (\"greedy\", 20.0), (\"eps\", 2.0)]\n        rows, res = [], {}\n        for st, q0 in cfg:\n            out = [need(ns, \"run\")(st, q0, 5000, 0.95, 0.2, s) for s in range(5)]\n            res[(st, q0)] = out\n            rows.append([st, q0, f\"{sum(o[0] for o in out) / 5:.4f}\", f\"{sum(o[1] for o in out)}/5\"])\n        ensure(sum(o[1] for o in res[(\"eps\", 0.0)]) == 0 and sum(o[1] for o in res[(\"boltzmann\", 0.0)]) == 0, \"ε-açgözlü ve Boltzmann tuzağa düşmeli\")\n        ensure(sum(o[1] for o in res[(\"greedy\", 2.0)]) == 5, \"Q0 = 2 ile açgözlü ajan yolu bulmalı\")\n        ensure(sum(o[0] for o in res[(\"greedy\", 2.0)]) > sum(o[0] for o in res[(\"greedy\", 20.0)]), \"çok iyimser Q0 keşfi uzatır\")\n        R_.table(\"Zincir, 5000 adım, 5 tohum\", [\"strateji\", \"Q0\", \"ort. ödül/adım\", \"doğru politika\"], rows,\n                 note=\"Sürekli ilerlemek 0.1/adım, yerel döngü 0.05/adım. Q0 = 1 tuzaktaki değere (0.05/(1−γ) = 1) eşit olduğu için yetmez; Q0 = 2 ≥ V* ≈ 1.57 olunca her denenmemiş eylem 'cazip' kalır.\")\n    R_.test(\"keşif stratejileri\", t)\n",
 "x3": "import numpy as np\nfrom labcheck import need, ensure\n\n\ndef basic():\n    p = np.zeros((2, 10, 10), dtype=int)\n    for t in range(12, 22):\n        for d in range(1, 11):\n            p[0, t - 12, d - 1] = 1 if (t == 12 and d not in (4, 5, 6)) or (13 <= t <= 16 and not 2 <= d <= 6) else 0\n            p[1, t - 12, d - 1] = 1 if t <= 17 or (t == 18 and d in (1, 9, 10)) else 0\n    return p\n\n\ndef check(ns, R_):\n    def t():\n        Q, pol = need(ns, \"mc_es\")(200000, 1)\n        ensure(Q.shape == (2, 10, 10, 2) and pol.shape == (2, 10, 10), \"şekiller\")\n        ref = basic()\n        agree = (pol == ref).mean()\n        ensure(agree > 0.88, f\"temel stratejiyle uyum %{100 * agree:.0f}\")\n        ensure((pol[:, 8:, :] == 0).all(), \"20–21'de dur\")\n        def show(m):\n            return [\"\".join(\"Ç\" if v else \"·\" for v in row) for row in m[::-1]]\n        rows = [[21 - i, a, b] for i, (a, b) in enumerate(zip(show(pol[0]), show(pol[1])))]\n        R_.table(\"Öğrenilen politika (Ç = çek, · = dur), sütunlar krupiye A,2…10\", [\"toplam\", \"kullanılabilir as yok\", \"kullanılabilir as\"], rows,\n                 note=f\"Temel stratejiyle uyum: %{100 * agree:.1f}. Uyumsuzluklar Q değerlerinin neredeyse eşit olduğu hücrelerde (ör. 12 – krupiye 3, yumuşak 18).\")\n    R_.test(\"MC keşifçi başlangıçlar\", t)\n",
 "x4": "import numpy as np\nfrom labcheck import need,ensure\n\n\ndef check(ns,R_):\n    def geometry():\n        track,step=need(ns,'track'),need(ns,'step')\n        m=track(0); r=np.random.default_rng(2)\n        ensure((m['W'],m['H'],len(m['road']),len(m['starts']),len(m['finish']))==(17,32,298,6,6),'sol kaynak geometrisi')\n        n=track(1)\n        ensure((n['W'],n['H'],len(n['road']),len(n['starts']),len(n['finish']))==(32,30,515,23,9),'sağ kaynak geometrisi')\n        z,reward,done,status=step(m,(8,2,1,0),7,r,0.)\n        ensure(status=='crash' and not done and z in m['starts'] and reward==-1,'duvar bölümü bitirmez; sıfır hızla başlangıca dön')\n        z,_,done,status=step(m,(14,28,4,0),4,r,0.)\n        ensure(done and status=='finish','son konum pist dışında olsa da çizgiyi önce geçmek bitiştir')\n        z,_,done,status=step(m,(3,0,0,0),8,r,1.)\n        ensure(z==(3,0,0,0) and not done,'iki ivme birlikte uygulanmamalı')\n    def learn():\n        control,trajectory=need(ns,'control'),need(ns,'trajectory')\n        o=control(100,0,3,.1,.1)\n        ensure(len(o['returns'])==100 and max(o['returns'])<0,'tam bölümlerin eksi adım getirileri')\n        ensure(all(np.isfinite(v).all() for v in o['Q'].values()),'log ağırlıkla sonlu Q')\n        for s,a in o['policy'].items():\n            ensure(a in need(ns,'actions')(o['model'],s),'öğrenilmiş eylem geçerli')\n        # The untrained proper prior must finish both full source geometries.\n        for k in (0,1):\n            m=need(ns,'track')(k)\n            for i in (0,len(m['starts'])//2,len(m['starts'])-1):\n                tr=trajectory(dict(model=m,policy={}),i,1,0.)\n                ensure(tr['finished'] and tr['steps']>0,'önsel sürücünün sonlanması')\n        tr=trajectory(o,0,1,0.,max_steps=1)\n        ensure(not tr['finished'] and tr['steps']==1,'kesme gerçek terminal gibi sunulmaz')\n    R_.test('Kaynak pistleri ve segment fiziği',geometry)\n    R_.test('Ağırlıklı MC, doğru eylemler ve yörüngeler',learn)\n",
 "x5": "import numpy as np\nfrom labcheck import need,ensure\n\n\ndef check(ns,R_):\n    def targets():\n        fn=need(ns,'prefix_targets')\n        N,D=fn([2,3,5],[999,0,4],0.)\n        ensure(np.allclose(N,[2,3,5]) and np.allclose(D,1),'gamma0 yalnız ilk ödül; mevcut eylemin oranı dışarıda')\n        N,D=fn([2,3,5],[999,2,3],1.)\n        ensure(np.allclose(N,[60,24,5]) and np.allclose(D,[6,3,1]),'gamma1 tam gelecek oranı')\n        N,D=fn([2,3,5],[999,2,3],.5)\n        ensure(np.allclose(N,[18.5,13.5,5]) and np.allclose(D,[2.5,2,1]),'bütün kısa önek katsayıları')\n    def control():\n        Q=np.zeros((3,2)); C=np.zeros_like(Q)\n        # At time 1 action1 is non-greedy even after its -2 update.\n        q,c=need(ns,'update_episode')(Q,C,[0,1],[0,1],[1,-2],[.5,.5],.5)\n        ensure(abs(c[0,0]-.5)<1e-12 and abs(q[0,0]-1)<1e-12,'uyumsuz sonraki eylem kısa öneki yok edemez')\n        q,c=need(ns,'update_episode')(Q,C,[0,1],[0,1],[1,-2],[.5,.5],1.)\n        ensure(c[0,0]==0 and c[1,1]==1,'gamma1 kuyruk uyuşmazsa önceki tam getiri ağırlığı0')\n    R_.test('İndirim duyarlı Q: üç gamma sınırı',targets)\n    R_.test('Politika değişimi ve kısa önekin korunması',control)\n",
 "x6": "import numpy as np\nfrom labcheck import need,ensure\n\n\ndef check(ns,R_):\n    def env():\n        step=need(ns,'windy_step')\n        ensure(step(38,3)==27,'hedefin sağında sola gitmek hedefin bir üstüne götürür; kaynak sütun')\n        class Noise:\n            def __init__(self,z):self.z=z\n            def integers(self,*args):return self.z\n        ensure([step(38,3,True,Noise(z)) for z in (-1,0,1)]==[37,27,17],'pozitif rüzgâr üç durum ve1/3 olasılık')\n        ensure(step(30,8)==30,'bekleme yalnız rüzgâr kadar hareket eder')\n        ensure(all(step(30,1,True,Noise(z))==31 for z in (-1,0,1)),'rüzgâr0 sütunlarına rastgele rüzgâr ekleme')\n    def model():\n        for actions,dist in [(4,15),(8,7),(9,7)]:\n            V,pi=need(ns,'optimal_windy')(actions,False)\n            ensure(abs(V[30]+dist)<1e-9,'bağımsız model minimum adım')\n            path,done=need(ns,'trajectory')(pi)\n            ensure(done and len(path)-1==dist,'optimal model yörüngesi')\n        V,pi=need(ns,'optimal_windy')(8,True)\n        ensure(abs(V[30]+10.9646090534948)<1e-8,'stokastik sekiz eylem model referansı')\n    def learn():\n        for actions,stochastic in [(8,False),(9,False),(8,True)]:\n            out=need(ns,'sarsa_windy')(4000,actions,stochastic,.5,.1,1)\n            ensure(out['Q'].shape==(70,actions) and np.isfinite(out['Q']).all(),'Q boyutu/sonluluk')\n            ensure(sum(out['lengths'])+out['unfinished']==4000,'tam bölümler ve bitmemiş parça toplamı')\n            ensure(len(out['history'])==4000 and np.all(np.diff(out['history'])>=0),'adım başına bölüm sayacı')\n            ensure(out['counts'].sum()==4000,'her TD adımında tek güncelleme')\n            ensure((out['Q'][37]==0).all(),'terminal Q0')\n    R_.test('Kaynak rüzgâr fiziği',env)\n    R_.test('4/8/9 eylem ve stokastik optimum',model)\n    R_.test('King hamleleri ve stokastik SARSA',learn)\n",
 "x7": "from labcheck import need\n\n\ndef check(ns, R_):\n    def task():\n        q_target = need(ns, 'q_target')\n        learn_q_chain = need(ns, 'learn_q_chain')\n        import math\n        S=[0,1,2];pi=[[.75,.25]]*3;b=[[.5,.5]]*3;Q=[[0.,0.],[2.,4.],[0.,0.]]\n        vals=[]\n        for action in [0,1]:\n            G,w=q_target(S,[0,action],[0.,0.],Q,pi,b,0,1,\"is\")\n            vals.append(w*G)\n        assert abs(sum(vals)/2-2.5)<1e-12  # The final sampled action's rho is essential.\n        for method in [\"expected\",\"cv\",\"tree\",\"sigma\"]:\n            for sigma in [0,.5,1]:\n                G,w=q_target(S,[0,0],[0.,0.],Q,pi,b,0,1,method,sigma)\n                assert abs(G-2.5)<1e-12 and w==1\n        # One real terminal step: no action A_T exists.\n        for method in [\"sarsa\",\"is\",\"expected\",\"cv\",\"tree\",\"sigma\"]:\n            G,w=q_target([0,1],[0],[2.],[[0.,0.],[0.,0.]],pi,b,0,4,method)\n            assert G==2. and w==1\n        learned=learn_q_chain(2000,4,.05,\"tree\",0,0)\n        assert len(learned)==5 and all(len(row)==2 for row in learned)\n        assert learned[4]==[0.,0.] and abs(learned[3][0]-2)<1e-10 and learned[3][1]==0\n        assert all(math.isfinite(x)for row in learned for x in row)\n    R_.test('n-adım Q: oran/ufuk/terminal sınırları', task)\n"
}

In [ ]:
import importlib, labcheck, checks_m11_5_sarsa_q as _c
importlib.reload(labcheck); importlib.reload(_c)
CHECKS = _c.CHECKS
print("hazır")

## Giriş

**Model olmadan kontrol:** $V^\pi$'den açgözlü politika çıkarmak için $P$ gerekir ($\arg\max_a R+\gamma\sum P V$). $Q^\pi(s,a)$ öğrenirsek $\pi'(s)=\arg\max_aQ(s,a)$ modelsiz olur. Bellman optimallik: $Q^*(s,a)=\mathbb E[r]+\gamma\sum_{s'}P(s'|s,a)\max_{a'}Q^*(s',a')$, $V^*=\max_aQ^*$, $\pi^*=\arg\max_aQ^*$.

**Monte Carlo kontrol:** politika değerlendirme (getirileri ortala) ile iyileştirmeyi dönüşümlü yap — politika yinelemesinin örneklemeli hâli. Her (s,a)'nın denenmesi için ya **keşifçi başlangıçlar** (rastgele (s,a)'dan başla) ya da ε-yumuşak politikalar.

**TD kontrol:**

| yöntem | hedef | tür |
|---|---|---|
| SARSA | $r+\gamma Q(s',a')$, $a'\sim\pi$ | politika üstü |
| beklenen SARSA | $r+\gamma\sum_{a'}\pi(a'|s')Q(s',a')$ | ikisi de olabilir |
| Q-öğrenme | $r+\gamma\max_{a'}Q(s',a')$ | politika dışı |

Q-öğrenme (Watkins 1989): `eylem seç ve uygula → r, s' gözle → n(s,a)++ → α = 1/n(s,a) → Q(s,a) += α(r + γ max Q(s',·) − Q(s,a)) → s ← s'`.

**Keşif mi sömürü mü:** hep açgözlü seçmek hiç denenmemiş eylemleri asla öğrenmez; hep rastgele seçmek öğrendiğini kullanmaz. ε-açgözlü (ε olasılıkla rastgele), Boltzmann ($\pi(a)\propto e^{Q(s,a)/\tau}$), iyimser başlangıç ($Q_0$ büyük: denenmemiş eylem cazip). GLIE: her çift sonsuz kez, politika sınırda açgözlü.

**Maksimizasyon yanlılığı:** gürültülü kestirimlerin maksimumu, maksimumun yanlı kestirimidir. Çift Q-öğrenme: iki tablo; $Q_1$ ile seç, $Q_2$ ile değerlendir: $Q_1(s,a)\leftarrow Q_1+\alpha[r+\gamma Q_2(s',\arg\max_{a'}Q_1(s',a'))-Q_1]$ (yazı-turaya göre roller değişir).

## 1 · Elle

<!-- çözüm -->

Q-öğrenme: hedef $1+0.9\cdot5=5.5$, $2+0.5\cdot3.5=3.75$. SARSA: $1+0.9\cdot3=3.7$, $2+0.5\cdot1.7=2.85$. Beklenen SARSA: $\pi=(0.1,0.9)$, $\mathbb E Q=4.8$, hedef 5.32, sonuç 3.66.

Boltzmann: $e^5/(e^3+e^5)=1/(1+e^{-2})$. ε-açgözlü: $1-\varepsilon+\varepsilon/|A|$.

İki bağımsız standart normalin maksimumu: $\max(X,Y)=\tfrac{X+Y}2+\tfrac{|X-Y|}2$, $X-Y\sim N(0,2)$, $\mathbb E|X-Y|=\sqrt2\sqrt{2/\pi}$ → $\mathbb E\max=1/\sqrt\pi$. Gerçek değerlerin maksimumu 0 iken kestirim 0.56!


Sitede bu aşama etkileşimli bir formdur. Burada değerleri aşağıdaki sözlüğe yazıp kontrol hücresini çalıştır.

**Tek güncelleme: $Q(s,a)=2$, $r=1$, $\gamma=0.9$, $Q(s',\cdot)=[3,5]$, $\alpha=0.5$**

- `ql`: Q-öğrenme sonrası $Q(s,a)$
- `sa`: SARSA, sonraki eylem $a'=0$
- `es`: Beklenen SARSA, ε = 0.2 (2 eylem)

**Keşif**

- `bz`: Boltzmann (τ = 1) ile $Q=[3,5]$'te eylem 1'in olasılığı
- `eg`: ε = 0.1, 4 eylem: açgözlü eylemin seçilme olasılığı

**Maksimizasyon yanlılığı**

- `mb`: İki eylemin gerçek değeri 0, kestirimler bağımsız N(0,1): $\mathbb E[\max]$ (= 1/√π)

In [ ]:
cevaplar = dict(
    ql=None,
    sa=None,
    es=None,
    bz=None,
    eg=None,
    mb=None,
)

In [ ]:
#@title Kontrol (1)
_A = {"ql":[3.75,0],"sa":[2.85,0],"es":[3.66,0],"bz":[0.8807970779778823,0.000001],"eg":[0.925,0],"mb":[0.5641895835477563,0.000001]}
_ok = 0
for k, (v, tol) in _A.items():
    g = cevaplar.get(k)
    good = g is not None and abs(float(g) - v) <= max(tol, 1e-9)
    _ok += good
    print(("✓ " if good else ("· " if g is None else "✗ ")) + k)
print(f"\n{_ok}/{len(_A)} doğru")

## 2 · Döngü (saf Python)

**Uçurum yürüyüşü** (Sutton Örnek 6.6), saf Python. 4×12 ızgara, durum `s = satır·12 + sütun`; başlangıç 36 (sol alt), hedef 47 (sağ alt); 37..46 uçurum. Eylemler 0 yukarı, 1 sağ, 2 aşağı, 3 sol; ızgara dışına çıkılmaz. Her adım −1; uçuruma basmak −100 ve başlangıca dönüş (bölüm sürer).

- `cliff_step(s, a)` → `(s2, r, done)`.
- `eps_greedy(Q, s, eps, rng)` — eşitlikleri rastgele boz.
- `sarsa(n_eps, alpha=0.5, eps=0.1, seed=0)` ve `qlearning(...)` → `(Q, getiriler)`; Q `48 × 4` liste.
- `greedy_return(Q, max_steps=100)` — açgözlü politikanın getirisi.

In [ ]:
%%writefile stage_b.py
import random
H, W = 4, 12
START, GOAL = 36, 47


def cliff_step(s, a):
    ...


def eps_greedy(Q, s, eps, rng):
    ...


def sarsa(n_eps, alpha=0.5, eps=0.1, seed=0):
    ...


def qlearning(n_eps, alpha=0.5, eps=0.1, seed=0):
    ...


def greedy_return(Q, max_steps=100):
    ...


print(greedy_return(qlearning(300)[0]))

In [ ]:
# Kontrol (2): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_b.py", CHECKS["b"])

## 3 · NumPy / pandas

**Maksimizasyon yanlılığı** (Sutton Örnek 6.7), NumPy, tüm deneyler aynı anda; yalnız bölüm döngüsü serbest.

A durumunda iki eylem: **sağ** → terminal, ödül 0; **sol** → B, ödül 0. B'de `n_b` eylem, her biri terminale gider ve ödül $N(-0.1,1)$. Doğru karar: sağ. γ = 1, ε-açgözlü (ε = 0.1), α = 0.1, Q başlangıçta 0, eşitlikler rastgele.

- `eps_greedy_probs(Q, eps)` — `(n, A)` olasılıklar (açgözlü eylem `argmax`).
- `max_bias(n_runs, n_eps, alpha=0.1, eps=0.1, n_b=10, seed=0, double=False)` — her bölümde sol seçen deney oranı `(n_eps,)`. Çift Q'da davranış $Q_1+Q_2$'ye göre ε-açgözlü; her deney kendi yazı-turasıyla hangi tabloyu güncelleyeceğini seçer.

In [ ]:
%%writefile stage_c.py
import numpy as np


def eps_greedy_probs(Q, eps):
    ...


def max_bias(n_runs, n_eps, alpha=0.1, eps=0.1, n_b=10, seed=0, double=False):
    ...


print(max_bias(1000, 100)[[0, 10, 99]])

In [ ]:
# Kontrol (3): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_c.py", CHECKS["c"])

## 4 · PyTorch

PyTorch, `float64`. `P` `(A, S, S)` satır-stokastik, `R` `(S, A)`.

- `q_value_iteration(P, R, gamma, tol=1e-10)` — $Q\leftarrow R+\gamma P\max_aQ$ (`einsum`).
- `sync_q_learning(P, R, gamma, n_iter, rate="rescaled", seed=0)` — **eşzamanlı** Q-öğrenme (üretici model): her yinelemede **tüm** (s,a) için $s'\sim P(\cdot|s,a)$ örnekle (`torch.multinomial`) ve hepsini birlikte güncelle. `rate`: `"1/n"` → $\alpha_n=1/n$; `"rescaled"` → $\alpha_n=1/(1+(1-\gamma)n)$.
- `greedy(Q)` → `(S,)`.

1/n neden yavaş? $\gamma$ yakın 1 iken önyüklemeli hedefin yanlılığı $\gamma$ hızında söner; eşit ağırlıklı ortalama erken yanlış hedefleri unutmaz — hata $\sim n^{-(1-\gamma)}$ (Even-Dar & Mansour).

In [ ]:
%%writefile stage_d.py
import torch


def q_value_iteration(P, R, gamma, tol=1e-10):
    ...


def sync_q_learning(P, R, gamma, n_iter, rate="rescaled", seed=0):
    ...


def greedy(Q):
    ...


g = torch.Generator().manual_seed(0)
P = torch.rand(3, 8, 8, generator=g, dtype=torch.float64); P /= P.sum(-1, keepdim=True)
R = torch.rand(8, 3, generator=g, dtype=torch.float64)
print((sync_q_learning(P, R, 0.9, 1000) - q_value_iteration(P, R, 0.9)).abs().max())

In [ ]:
# Kontrol (4): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_d.py", CHECKS["d"])

## 5 · Vektörize + hız

1000 uçurum ortamı paralel (PyTorch). Q `(n_envs, 48, 4)`.

- `step(s, a)` — vektörel ortam: `(s2, r, done)` (uçurum → 36, −100).
- `batched_control(n_envs, n_steps, alpha, eps, method="q", seed=0)` — `method ∈ {"q", "sarsa", "expected"}`; her adımda her ortam bir adım atar; hedefe varan ortam başa döner (sonraki eylem yeniden seçilir). Çıktı `(Q, rew)`: `rew[t]` adım $t$'deki ortalama ödül.
- `greedy_returns(Q, max_steps=100)` → `(n_envs,)` açgözlü politikaların getirileri.

Adım döngüsü serbest; ortam döngüsü yok.

In [ ]:
%%writefile stage_e.py
import torch
H, W, START, GOAL = 4, 12, 36, 47


def step(s, a):
    ...


def batched_control(n_envs, n_steps, alpha, eps, method="q", seed=0):
    ...


def greedy_returns(Q, max_steps=100):
    ...


Q, rew = batched_control(100, 2000, 0.5, 0.1)
print(greedy_returns(Q)[:10])

In [ ]:
# Kontrol (5): kodunu çalıştırır ve sitedeki testleri uygular
labcheck.run_file("stage_e.py", CHECKS["e"])

## Ek alıştırmalar

Beş aşamanın dışındaki pratikler; her biri kendi kontrolüyle.

### E1 · Rüzgârlı ızgara dünyası ile SARSA

**Rüzgârlı ızgara** (Sutton Örnek 6.5): 7×10, başlangıç (3,0), hedef (3,7); sütun rüzgârı `[0,0,0,1,1,1,2,2,1,0]` ajanı yukarı iter (hareket sütununun rüzgârı). Her adım −1. SARSA, ε = 0.1, α = 0.5.

- `windy_step(r, c, a)` → `(r2, c2)`.
- `sarsa_windy(n_steps, seed=0)` → `(Q, bitirilen bölüm sayısı listesi her adımda)`.
- `greedy_len(Q)` — açgözlü yol uzunluğu (en fazla 100).

In [ ]:
%%writefile extra_x1.py
import random

WIND = [0, 0, 0, 1, 1, 1, 2, 2, 1, 0]


def windy_step(r, c, a):
    ...


def sarsa_windy(n_steps, seed=0):
    ...


def greedy_len(Q):
    ...


Q, ep = sarsa_windy(8000)
print(ep[-1], greedy_len(Q))

In [ ]:
# Kontrol (E1)
labcheck.run_file("extra_x1.py", CHECKS["x1"])

### E2 · Keşif: zincir ortamında ε-açgözlü, Boltzmann, iyimser başlangıç

**Zincir:** N = 10 durum. Eylem 0: başa dön (zaten 0'daysan ödül 0.05); eylem 1: ilerle (N−1'de ödül 1 ve başa dön). γ = 0.95: sürekli ilerlemek daha iyi ($V\approx1.57$ vs 1.0). Q-öğrenme, α = 0.2, 5000 adım.

- `run(strategy, Q0=0.0, n_steps=5000, gamma=0.95, alpha=0.2, seed=0)` → `(ortalama ödül, açgözlü politika her durumda ilerliyor mu)`; `strategy ∈ {"eps", "boltzmann", "greedy"}` (ε = 0.1, τ = 0.1; eşitlikler rastgele).

Sonuç: küçük yerel ödül ε-açgözlü ve Boltzmann'ı tuzağa düşürür (dokuz ardışık rastgele "ilerle" gerekir); $Q_0$ yeterince iyimser (≥ gerçek en iyi değer) açgözlü ajan yolu bulur; çok iyimser $Q_0$ ise keşfi uzatır.

In [ ]:
%%writefile extra_x2.py
import math
import random
N = 10


def chain_step(s, a):
    if a == 0:
        return 0, (0.05 if s == 0 else 0.0)
    if s == N - 1:
        return 0, 1.0
    return s + 1, 0.0


def run(strategy, Q0=0.0, n_steps=5000, gamma=0.95, alpha=0.2, seed=0):
    ...


print(run("eps"), run("greedy", Q0=2.0))

In [ ]:
# Kontrol (E2)
labcheck.run_file("extra_x2.py", CHECKS["x2"])

### E3 · Monte Carlo kontrol: keşifçi başlangıçlarla blackjack

**MC keşifçi başlangıçlar** (Sutton Şekil 5.2): her bölüm rastgele (as, toplam 12–21, krupiye 1–10) durumu ve rastgele eylemle başlar, sonra mevcut politika izlenir; her ziyaret edilen (s,a) için getiri ortalanır ve politika o durumda açgözlü yapılır.

- `mc_es(n, seed=0)` → `(Q (2,10,10,2), policy (2,10,10))`, eylem 1 = kart çek. Başlangıç politikası: 20–21'de dur, yoksa çek.

In [ ]:
%%writefile extra_x3.py
import random
import numpy as np


def draw(rng):
    return min(rng.randint(1, 13), 10)


def mc_es(n, seed=0):
    ...


Q, pol = mc_es(20000)
print(pol[0, ::-1])

In [ ]:
# Kontrol (E3)
labcheck.run_file("extra_x3.py", CHECKS["x3"])

### E4 · İki yarış pistinde politika dışı MC — Sutton5.12

# Yarış pistinde politika dışı Monte Carlo

Kaynak: Sutton ve Barto (2018), s.111–112, Alıştırma5.12 ve Şekil5.5. İki kaynak pistinin bütün hücreleri `TRACKS` içinde görev verisi olarak yeniden kodlandı; fotoğraf/görsel kullanılmadı.

`step`, `control` ve `trajectory` işlevlerini tamamlayın. Durum `(x,y,vx,vy)`, dokuz nominal eylem `(−1,0,+1)²` ivmedir. Her hız 0…4; ikisi birlikte yalnız başlangıç çizgisinde0 olabilir. Geçersiz son hız üreten eylemi ele, kırparak başka eyleme dönüştürme. Başlangıç düzgün rastgele, hız0; her adımın ödülü−1. Hareket parçasının duvarla ve bitiş çizgisiyle ilk kesişimini sırayla denetle: bitiş geçildiyse terminal, önce duvar varsa rastgele başlangıca sıfır hızla dön; bölüm sürer. Hücre köşesinden duvara temas çarpışma sayılır. Yeşil hücre merkezlerinin düşey doğrusu bitiştir; bu ayrık geometri için açık seçimimizdir.

Eğitimde %10 olasılıkla iki ivme birlikte0 olur. Hedef tutarlı bağ kuralıyla açgözlü, davranış ε-greedy. Davranış olasılığını **eylem seçildiği anda** sakla. Geri geçişte önce `G=−1+G`, ağırlıklı Q ortalaması ve hedefi güncelle; eylem yeni hedefle uyuşmazsa dur, yoksa `W/=b`. Taşmayı önlemek için `logC=logaddexp(logC,logW)` ve oran `exp(logW−logC)` kullan; önem ağırlığını kırpma.

Verilen düşük hızlı hücre-uzaklığı önseli yalnız düzgün bir başlangıç sürücüsüdür. `Q=−1000` ve önsel bağ kuralını belirtin. Başlangıç çizgisinde iki son hızı0 yapan eylemler hedefte elenir, davranışta yasaldır. Sıfır hızda beklemek aynı duruma−1 ile döner; başka başlangıç-hız durumları için bu ek sürücü kuralıdır. Tam model referansı bütün yasal eylemleri kullanır. Sayısal denetimde bu iki haritanın bütün durumlarında eleme optimum V değerlerini değiştirmedi. Güvenlik bütçesini aşan bölümü terminal saymak yerine hata üretin.

İki pist için en az50000 bölüm eğitin. Başlangıcın sol/orta/sağ noktalarından birkaç yörüngeyi `noise=0` ile gösterin; çarpışma yeniden başlatması korunur. Örneğin:

```python
result = control(50000, which=0, seed=1)
traces = [trajectory(result,i,noise=0) for i in (0,3,5)]
print([(t['finished'],t['steps']) for t in traces])
```

`racetrackmc` deneyi iki pistte gerçek eğitimi ve bağımsız tam-model değer yinelemesiyle bulunan optimumu ayrı gösterir. Sonlu MC çalıştırması kaynak sorunun optimal politika hedefi için bir kestirimdir; tüm hücrelerde optimal olduğunu kanıtlamaz. Öğrenilmiş başlangıç değeri ile model optimumunu karşılaştırın. Python ve JS tohumları aynı yörüngeyi üretmez. Kontrol küçük bir eğitim bütçesiyle fizik ve güncelleme hatalarını yakalar; uzun deney ölçümleri üretim kaydındadır.

In [ ]:
%%writefile extra_x4.py
import math
import numpy as np

TRACKS = [['###.............F', '##..............F', '##..............F', '#...............F', '................F', '................F', '..........#######', '.........########', '.........########', '.........########', '.........########', '.........########', '.........########', '.........########', '#........########', '#........########', '#........########', '#........########', '#........########', '#........########', '#........########', '#........########', '##.......########', '##.......########', '##.......########', '##.......########', '##.......########', '##.......########', '##.......########', '###......########', '###......########', '###SSSSSS########'], ['################...............F', '#############..................F', '############...................F', '###########....................F', '###########....................F', '###########....................F', '###########....................F', '############...................F', '#############..................F', '##############................##', '##############.............#####', '##############............######', '##############..........########', '##############.........#########', '#############..........#########', '############...........#########', '###########............#########', '##########.............#########', '#########..............#########', '########...............#########', '#######................#########', '######.................#########', '#####..................#########', '####...................#########', '###....................#########', '##.....................#########', '#......................#########', '.......................#########', '.......................#########', 'SSSSSSSSSSSSSSSSSSSSSSS#########']]
ACCEL = [(x,y) for x in (-1,0,1) for y in (-1,0,1)]


def track(which=0):
    rows = TRACKS[which]
    H, W = len(rows), len(rows[0])
    road = {(x,H-1-j) for j,row in enumerate(rows) for x,c in enumerate(row) if c != '#'}
    starts = sorted((x,y,0,0) for j,row in enumerate(rows) for x,c in enumerate(row) if c == 'S' for y in [H-1-j])
    finish = {(x,H-1-j) for j,row in enumerate(rows) for x,c in enumerate(row) if c == 'F'}
    return dict(W=W,H=H,road=road,starts=starts,finish=finish)


def actions(m, s):
    x,y,vx,vy = s
    return [a for a,(dx,dy) in enumerate(ACCEL)
            if 0 <= vx+dx < 5 and 0 <= vy+dy < 5
            and (vx+dx or vy+dy or (x,y,0,0) in m['starts'])]


def crossing(m, x, y, vx, vy):
    # Segment/grid supercover: corner contact with either wall is a crash.
    if vx == vy == 0:
        return (x,y,vx,vy), 'road'
    xf = next(iter(m['finish']))[0]
    lo,hi = min(p[1] for p in m['finish'])-.5, max(p[1] for p in m['finish'])+.5
    tf = (xf-x)/vx if vx else math.inf
    if not (0 < tf <= 1 and lo <= y+tf*vy <= hi):
        tf = math.inf
    cx,cy,tx,ty = x,y,.5/vx if vx else math.inf,.5/vy if vy else math.inf
    while True:
        t = min(tx,ty)
        if tf <= min(t,1):
            return None, 'finish'
        if t > 1:
            return (x+vx,y+vy,vx,vy), 'road'
        if abs(tx-ty) < 1e-12:
            if not {(cx+1,cy),(cx,cy+1),(cx+1,cy+1)} <= m['road']:
                return None, 'crash'
            cx,cy,tx,ty = cx+1,cy+1,tx+1/vx,ty+1/vy
        elif tx < ty:
            cx,tx = cx+1,tx+1/vx
            if (cx,cy) not in m['road']: return None, 'crash'
        else:
            cy,ty = cy+1,ty+1/vy
            if (cx,cy) not in m['road']: return None, 'crash'


def guide(m, s):
    # Proper low-speed prior; not the solution of the stochastic full MDP.
    if 'distance' not in m:
        d = {}
        for y in reversed(range(m['H'])):
            for x in reversed(range(m['W'])):
                if (x,y) in m['finish']: d[x,y] = 0
                elif (x,y) in m['road']: d[x,y] = 1+min(d.get((x+1,y),1e6),d.get((x,y+1),1e6))
        m['distance'] = d
    scores=[]
    for a in actions(m,s):
        vx,vy=s[2]+ACCEL[a][0],s[3]+ACCEL[a][1]
        if vx == vy == 0: continue
        z,status=crossing(m,*s[:2],vx,vy)
        score=-1e6 if status=='finish' else 1e6 if status=='crash' else 1000*(max(0,vx-1)+max(0,vy-1))+m['distance'][z[:2]]
        scores.append((score,a))
    return min(scores)[1]


def step(m, s, a, rng, noise=.1):
    # TODO: joint acceleration failure, segment crossing, crash reset.
    raise NotImplementedError


def control(episodes=1000, which=0, seed=1, epsilon=.1, noise=.1):
    # TODO: soft behavior + weighted off-policy MC, saved action probabilities.
    raise NotImplementedError


def trajectory(result, start=0, seed=1, noise=0., max_steps=2000):
    # TODO: evaluate without acceleration noise; report truncation explicitly.
    raise NotImplementedError

In [ ]:
# Kontrol (E4)
labcheck.run_file("extra_x4.py", CHECKS["x4"])

### E5 · İndirim duyarlı Q kontrolü — Sutton5.14

# İndirim duyarlı politika dışı kontrol

Kaynak: Sutton ve Barto (2018), s.115, Alıştırma5.14. Bölüm5.8 içindeki durum-değeri karışımını Q kontrolüne uyarlayın. `prefix_targets` sonlu bir bölümün her t noktası için N ağırlıklı getiri payını ve D toplam önek ağırlığını döndürsün. Mevcut eylem koşullandığı için çarpım t+1 ile başlar. Bütün katsayılar ve türetim M11.4, “İndirim ve karar başına oranlar” bölümündedir.

`update_episode` geriye işler. Her t için güncel `argmax Q` hedefini kullanarak bütün kısa önekleri yeniden hesaplar, ardından C ve Q günceller. Davranış olasılıklarını bölümde seçildikleri haliyle kullanır. Sonraki bir eylem hedef dışı olsa bile daha kısa önekler katkı verebilir; γ<1 iken dış geriye döngüyü erkenden bitirmeyin. Bu açık O(T²) uygulama, aynı geçişte değişen hedefle önbellek uyumsuzluğunu önler. Sabit hedef için O(T) N/D yinelemesini ayrıca türetin.

γ=0 tek ödülü, γ=1 tam son-ek ağırlıklı MC kontrolünü vermelidir. N/D tanımsızsa (D=0) o güncellemeyi atlayın. Bu alıştırma yeni kontrol yönteminin bütün MDP sınıflarında yakınsadığı iddiası değildir; verilen güncellemenin cebirsel uyarlamasıdır.

In [ ]:
%%writefile extra_x5.py
import numpy as np


def prefix_targets(rewards, ratios, gamma):
    # TODO: finite-horizon mixture of flat returns; Q excludes current-action ratio.
    raise NotImplementedError


def update_episode(Q,C,states,actions,rewards,behavior,gamma):
    # TODO: reverse updates with the current greedy target; keep short prefixes.
    raise NotImplementedError

In [ ]:
# Kontrol (E5)
labcheck.run_file("extra_x5.py", CHECKS["x5"])

### E6 · King hamleleri ve stokastik rüzgâr — Sutton6.9/6.10

# Şahın hamleleri ve stokastik rüzgâr

Kaynak: Sutton ve Barto (2018), s.130–131, Alıştırma6.9 ve6.10. Eski x1 dört eylemli program korunur. Burada aynı7×10 ızgarada önce dört ana yöne kuzeydoğu,kuzeybatı,güneydoğu,güneybatı ekleyen sekiz şah hamlesiyle, sonra dokuzuncu bekleme eylemiyle yeniden çözün. Ardından sekiz hamleli modeli stokastik rüzgâra taşıyın.

Başlangıç(3,0),hedef(3,7),her geçiş−1,γ1,Q0. Rüzgâr sütunları0,0,0,1,1,1,2,2,1,0. **Kaynak sütunun** yukarı rüzgârını eylemle birlikte ekleyip son konumu sınırda kırpın. Stokastik çeşitte yalnız pozitif rüzgârlı sütunlarda etkili rüzgâr temel−1/temel/temel+1,her biri1/3 olur. Örneğin hedefin sağından sola gidince hedefin kendisi,bir üstü,iki üstü eşit olasılıklıdır. Rüzgâr0 sütunlarını rastgele aşağı/yukarı kaydırmayın.

`windy_step`,`optimal_windy`,`sarsa_windy`,`trajectory` işlevlerini tamamlayın. SARSA,A' eylemini mevcut Q güncellenmeden **önce** seçer. Hedefe varınca Q(terminal)=0;devam eden bölümde durum ve önce seçilmiş eylemi devret. Sabit toplam TD adım bütçesinin sonunda bitmemiş bölümü sahte terminal saymayın. Çıktı Q,sayım matrisi,her adımdaki tamamlanan bölüm sayısı,bölüm uzunlukları ve bitmemiş adım sayısını taşısın.

Temel kaynak deneyi8000 adım,α0.5,ε0.1. Kendi tohumunuzu yazın;Python/JS rastgele üreteçleri aynı sayısal tohumu aynı yollara dönüştürmez. Stokastik örnekte100000 adım veα0.1 gibi ek bütçeleri karşılaştırın. Sonlu eğitimde açgözlü politikanın döngüye girebilmesi mümkündür;`trajectory` bütçe sonunda gerçek bitiş bayrağını vermelidir.

Bağımsız model referansını da hesaplayın. Belirlenimci minimum dört eylemde15,sekiz ve dokuz eylemde7. Hedef yatay7 hücre uzakta olduğu için bekleme7 altına indiremez. Stokastik sekiz eylemde optimum beklenen adım10.9646090534948. Bu sayıları öğrenilmiş politika diye sunmayın:modeli bilinmeyen SARSA yalnız örnekleri kullanır. Hem model hem sonlu öğrenim yörüngelerini windytd deneyinde karşılaştırın. Uzunluk ve getirinin burada işaretleri ters;her adım−1.

In [ ]:
%%writefile extra_x6.py
import numpy as np

WIND = np.array([0,0,0,1,1,1,2,2,1,0])
MOVES = [(-1,0),(0,1),(1,0),(0,-1),(-1,-1),(-1,1),(1,1),(1,-1),(0,0)]
START, GOAL = 30, 37


def windy_step(s,a,stochastic=False,rng=None):
    # TODO: kaynak sütunun rüzgârı, pozitif sütunlarda üç olası değişim.
    raise NotImplementedError


def optimal_windy(actions=8,stochastic=False):
    # TODO: bağımsız Bellman referansı; hedef değeri0.
    raise NotImplementedError


def sarsa_windy(steps=8000,actions=8,stochastic=False,alpha=.5,epsilon=.1,seed=1):
    # TODO: A' güncellemeden önce seçilsin; bitmeyen son parça terminal sayılmasın.
    raise NotImplementedError


def trajectory(policy,stochastic=False,seed=1,budget=1000):
    # TODO: yörünge ve gerçek sonlanma bayrağı.
    raise NotImplementedError

In [ ]:
# Kontrol (E6)
labcheck.run_file("extra_x6.py", CHECKS["x6"])

### E7 · n-adım Q: IS/CV/Tree/Qσ ve terminal sınırları

Kaynak:Sutton ve Barto (2018),s.147/149/151/154/156. n-adım Q hedeflerinin terminal ve önem oranı sınırlarını çalışan kodla inceleyin. Bu ek,7.7 sözde-kod sorusunun da programlanmış karşılığıdır.

`q_target(states,actions,rewards,Q,pi,behavior,tau,n,kind="is",sigma=0.5,gamma=1) → (G,weight)`;durum listesi terminali içerir,eylem listesi yalnız gerçek eylemleri içerir. `kind`:on-policy `sarsa`,tam oranlı`is`,son-beklenti`expected`,kontrol-değişkenli`cv`, `tree`,kitabın(7.17)kontrol-değişkenli`sigma` biçimi. Bütün tablolara durum indeksiyle erişilir. Ufuk nonterminalse `A_h` saklıdır;terminalde `A_T` yoktur ve aranmaz.

- Ordinary IS oranı τ+1…min(τ+n,T−1),Expected içinbirazkısadır. Kaynak s.149 kutusunun son indeks eksiği düzeltilir.
- CV/Tree/Qσ iç katsayıları kendisi işler;dış weight1 döndürür. Ufuk nonterminalse G_h=Q(S_h,A_h),terminalse0;son geçişin hedefiR_T. Kaynaktaki G0 satırı nonterminal ufukta uygulanmaz.
- `learn_q_chain(...)`:dört durum,iki eylem,hep sonraki duruma geçiş;son eylem0 ödül2,eylem1 ödül0. Sabitπ=(.75,.25),b=(.5,.5). Gerçek son Q=(2,0),önceki üç durumun her iki Q değeri1.5. Varyans nedeniyle sonlu sabitα tablo tam modele eşit olmak zorunda değildir.

Bu kod sabit politika tahmini içindir;kontrol yordamındaki ε-greedy/greedy iyileştirme ayrıca uygulanır. Kitabın kontrol-değişkenliQσ,sigma1 için7.14 biçimine iner;düz örneklemeliSARSA7.4 ile her yolda birebir aynı değildir. İki Qσ biçimini karıştırmayın.

In [ ]:
%%writefile extra_x7.py
def q_target(states, actions, rewards, Q, pi, behavior, tau, n, kind="is", sigma=0.5, gamma=1.0):
    # TODO: doğru ufuk, terminal ve önem oranı.
    return 0.0, 1.0


def learn_q_chain(episodes=10000, n=4, alpha=0.05, kind="sigma", sigma=0.5, seed=0):
    # TODO: verilen dört durumlu zincirde Q tahmini.
    return [[0.0, 0.0] for _ in range(5)]

In [ ]:
# Kontrol (E7)
labcheck.run_file("extra_x7.py", CHECKS["x7"])

## Makale ödevi: Watkins & Dayan (1992): Q-öğrenmenin yakınsaması

[Watkins & Dayan — Q-learning (Machine Learning 1992)](https://doi.org/10.1007/BF00992698)

Makaledeki yakınsama koşullarını (her (s,a) sonsuz kez, $\sum\alpha=\infty$, $\sum\alpha^2<\infty$, sınırlı ödül) deneysel olarak sına: rastgele MDP'lerde Q*'ı değer yinelemesiyle bul, Q-öğrenmeyi (a) ε-açgözlü, (b) bazı (s,a) çiftlerini hiç denemeyen davranış politikasıyla, (c) $\alpha_n=1/n$, $1/n^{0.6}$ ve $1/(1+(1-\gamma)n)$ ile çalıştır.


**Adımlar**

1. Bu modülün d aşamasındaki q_value_iteration'ı referans al.
2. Hata ‖Q − Q*‖∞'u log-log çiz; γ = 0.9 ve 0.99 için 1/n'nin yavaşlığını göster (Even-Dar & Mansour).

**Kabul ölçütleri**

- (b)'de denenmeyen çiftlerin Q değeri değişmez ve açgözlü politika yanlış olabilir.
- 1/n ile hata γ = 0.99'da 10⁵ adımda bile büyük kalır.

## Makale ödevi: van Hasselt (2010): Çift Q-öğrenme

[van Hasselt — Double Q-learning (NeurIPS 2010)](https://papers.nips.cc/paper/2010/hash/091d584fced301b442654dd8c23b3fc9-Abstract.html)

Makalenin rulet ve 3×3 ızgara dünyası deneylerini yeniden üret. Izgarada her adımın ödülü −12 ya da +10 (eşit olasılık), hedefte +5; ε = 1/√n(s) ve α = 1/n(s,a). Q-öğrenme ile çift Q-öğrenmenin adım başı ortalama ödülünü ve başlangıç durumunun max Q kestirimini karşılaştır.


**Adımlar**

1. c aşamasındaki tablo düzenini kullan: iki Q tablosu, her adımda yazı-tura ile hangisinin güncelleneceğini seç.
2. 10 000 deney ortalaması.

**Kabul ölçütleri**

- Q-öğrenme max Q(s₀)'ı aşırı kestirir, çift Q-öğrenme hafifçe eksik kestirir (makale Şekil 2).

## Kaynaklar

- [Sutton & Barto — Reinforcement Learning: An Introduction (2. baskı), Bölüm 5.3 ve 6.4–6.7](http://incompleteideas.net/book/the-book-2nd.html) — CC BY-NC-ND 2.0
- [Watkins & Dayan — Q-learning (Machine Learning 1992)](https://doi.org/10.1007/BF00992698) — ©
- [van Hasselt — Double Q-learning (NeurIPS 2010)](https://papers.nips.cc/paper/2010/hash/091d584fced301b442654dd8c23b3fc9-Abstract.html) — ©
- [Even-Dar & Mansour — Learning Rates for Q-learning (JMLR 2003)](https://www.jmlr.org/papers/v5/evendar03a.html) — ©
- [Singh,Jaakkola,Littman,Szepesvári (2000) — SARSA yakınsaması,Teorem1](https://sites.ualberta.ca/~szepesva/papers/singh98convergence.pdf) — ©
- [van Seijen,van Hasselt,Whiteson,Wiering (2009) — Expected SARSA](https://www.ai.rug.nl/~mwiering/GROUP/ARTICLES/expected_sarsa.pdf) — ©